data preporcessingdan 4 ta basic ca xar biriga 5 ta follow up tuzing

yangi dataset toping va classification va regression alohida aloxida k foldni toping

In [10]:
import pandas as pd 
df = pd.read_csv(r"C:\Users\PC\MLFoundation\Sherzod_dev\2-oy_Model_Building\Data\Social_Network_Ads.csv")
df.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   User ID          400 non-null    int64 
 1   Gender           400 non-null    object
 2   Age              400 non-null    int64 
 3   EstimatedSalary  400 non-null    int64 
 4   Purchased        400 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 15.8+ KB


In [12]:
df.drop(['User ID'], axis=1, inplace=True, errors='ignore')
df.head()

,Gender,Age,EstimatedSalary,Purchased
0,Male,19,19000,0
1,Male,35,20000,0
2,Female,26,43000,0
3,Female,27,57000,0
4,Male,19,76000,0


In [13]:
def datalarni_toldir(data):
    for col in data.columns:
        if data[col].isnull().any():
            if data[col].dtype == "object":
                data[col].fillna(data[col].mode()[0], inplace=True)
            else:
                data[col].fillna(data[col].mean(), inplace=True)
    return data

df = datalarni_toldir(df)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Gender           400 non-null    object
 1   Age              400 non-null    int64 
 2   EstimatedSalary  400 non-null    int64 
 3   Purchased        400 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 12.6+ KB


In [14]:
from sklearn.preprocessing import LabelEncoder

def labella_funksiya(data):
    le = LabelEncoder()
    for ustun in data.columns:
        # Faqat matnli (object) ustunlarni tanlaymiz
        if data[ustun].dtype == 'object':
            # Har bir matnli ustunni raqamga o'giramiz
            data[ustun] = le.fit_transform(data[ustun].astype(str)) # astype(str) xatolikni oldini oladi
    return data

# Funksiyani chaqiramiz
df = labella_funksiya(df)
df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Gender           400 non-null    int64
 1   Age              400 non-null    int64
 2   EstimatedSalary  400 non-null    int64
 3   Purchased        400 non-null    int64
dtypes: int64(4)
memory usage: 12.6 KB


In [15]:
from sklearn.preprocessing import StandardScaler

# Classification target bo'lgan Purchased ustunini scale qilmaymiz
raqamli_ustun = [col for col in df.select_dtypes(include=['int64', 'float64']).columns if col != 'Purchased']
standard_scaler = StandardScaler()
df[raqamli_ustun] = standard_scaler.fit_transform(df[raqamli_ustun])

# Purchased ni discrete class sifatida saqlab qolamiz
df['Purchased'] = df['Purchased'].astype(int)

df.head()

,Gender,Age,EstimatedSalary,Purchased
0,1.020204,-1.781797,-1.490046,0
1,1.020204,-0.253587,-1.460681,0
2,-0.980196,-1.113206,-0.785290,0
3,-0.980196,-1.017692,-0.374182,0
4,1.020204,-1.781797,0.183751,0


## model Selection

In [16]:
from sklearn.model_selection import KFold, cross_val_score
target_column_reg = 'EstimatedSalary'
X_reg = df.drop(target_column_reg, axis=1)
y_reg = df[target_column_reg]

kf = KFold(n_splits = 5, shuffle=True, random_state=42)

In [17]:
from sklearn.linear_model import LogisticRegression, LinearRegression

linear_model = LinearRegression()

scores_reg = cross_val_score(linear_model, X_reg, y_reg, cv=kf, scoring='r2')
print("Regression R2 natijalar:", scores_reg)
print("O'rtacha R2:", scores_reg.mean())


Regression R2 natijalar: [0.06682126 0.13510211 0.21091337 0.08115996 0.08264806]
O'rtacha R2: 0.11532895098949354


### Classification orqali baholash xuddi shu k-fold usulida
Bu yerda classification uchun target o'zgarishi kerak (qaysi foydalanuvchi sotib olganini aniqlaymiz).

In [18]:
from sklearn.model_selection import StratifiedKFold

logistic_model = LogisticRegression(random_state=42)

# Classification uchun nishonni (target y) Purchased deb olamiz
target_class = 'Purchased'
y_class = df[target_class]
X_class = df.drop(target_class, axis=1)

# Classification uchun stratified k-fold ishlatamiz
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_class = cross_val_score(logistic_model, X_class, y_class, cv=skf, scoring='accuracy')

print("Classification natijalari (Accuracy):", scores_class)
print("O'rtacha Accuracy score:", scores_class.mean())

Classification natijalari (Accuracy): [0.8625 0.825  0.8625 0.8375 0.8375]
O'rtacha Accuracy score: 0.845


In [19]:
# K-Fold natijalarini jadval ko'rinishida chiqarish (tabulate)
results_df = pd.DataFrame([
    {
        'Task': 'Regression',
        'Metric': 'R2',
        'Fold Scores': ', '.join([f'{s:.4f}' for s in scores_reg]),
        'Mean Score': round(scores_reg.mean(), 4)
    },
    {
        'Task': 'Classification',
        'Metric': 'Accuracy',
        'Fold Scores': ', '.join([f'{s:.4f}' for s in scores_class]),
        'Mean Score': round(scores_class.mean(), 4)
    }
])

try:
    from tabulate import tabulate
    print(tabulate(results_df, headers='keys', tablefmt='github', showindex=False))
except ImportError:
    print('tabulate paketi topilmadi. Quyidagi oddiy jadvalni ko\'ring:')
    print(results_df.to_string(index=False))

| Task           | Metric   | Fold Scores                            |   Mean Score |
|----------------|----------|----------------------------------------|--------------|
| Regression     | R2       | 0.0668, 0.1351, 0.2109, 0.0812, 0.0826 |       0.1153 |
| Classification | Accuracy | 0.8625, 0.8250, 0.8625, 0.8375, 0.8375 |       0.845  |
